# Vector Database

In [ ]:
from pathlib import Path

from langchain_chroma import Chroma
from langchain_community.document_loaders import WikipediaLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from more_itertools import chunked
from tqdm import tqdm

from chain_reaction.config import APIKeys

# Load API Keys
api_keys = APIKeys()

# Configure local data directory
path_parts = Path.cwd().parts
root_dir_index = path_parts.index("chain-reaction")
root_dir = Path(*path_parts[: root_dir_index + 1])
data_dir = root_dir / "data"
data_dir.mkdir(exist_ok=True)

## Gather Wikipedia articles for database

In [ ]:
topics = [
    "Retrieval Augmented Generation",
    "Agentic RAG",
    "Corrective RAG",
    "Adaptive RAG",
    "Semantic search",
    "Dot product",
    "Vector Database",
    "Embeddings",
    "Large Language Models",
    "LLM Hallucinations",
    "Text Search",
]

all_docs = []
for topic in topics:
    docs = WikipediaLoader(query=topic, load_max_docs=1, doc_content_chars_max=100_000).load()
    print(f"Downloaded {len(docs)} for {topic}")
    all_docs.extend(docs)

## Chunk documents

In [ ]:
# Average document length
doc_lens = [len(doc.page_content) for doc in all_docs]
sum(doc_lens) / len(doc_lens)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(all_docs)
len(chunks)

## Initialize Text Embedding Model

In [ ]:
# Map model name → embedding function factory
EMBEDDING_REGISTRY: dict[str, callable] = {
    "openai-small": lambda: OpenAIEmbeddings(model="text-embedding-3-small", api_key=api_keys.openai),
    "openai-large": lambda: OpenAIEmbeddings(model="text-embedding-3-large", api_key=api_keys.openai),
    # add models here
}
embedding_model_name = "openai-small"
embedding_model = EMBEDDING_REGISTRY[embedding_model_name]()

In [ ]:
# Simple test embedding
test_embedding = embedding_model.embed_query("Test query")
len(test_embedding)

# Initialize and populate VectorDB

In [ ]:
# Initialize persistent Chroma database
vector_store = Chroma(
    collection_name="rag-wiki",
    embedding_function=embedding_model,
    persist_directory=data_dir / "chroma_db",
    collection_metadata={"embedding_model": embedding_model_name},
)

In [ ]:
# Ingest documents async in batches
batch_size = 50
for doc_batch in tqdm(
    chunked(chunks, n=50),
    total=len(chunks) // batch_size + bool(len(chunks) % batch_size),
    desc="Ingesting document chunks",
    unit="batch",
    leave=False,
):
    await vector_store.aadd_documents(doc_batch)

In [ ]:
# Number of documents in collection
num_documents = vector_store._client.get_collection("rag-wiki").count()
print(f"# Document chunks: {num_documents:>6,}")